# 核函数高阶方法 vs 低阶方法：Ramsey 测频精度对比

**目标**：评估 `KernelEstimator` 的 order=1/2/3 Volterra 核对 Ramsey 瞬态测频精度的影响。

**核心问题**：高阶核函数能否改善频率测量精度？在什么条件下有意义？

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time, math
from sqc.config import CONFIG
from sqc.reconstruction.kernel import KernelEstimator, KernelResult
from sqc.control.sequence import create_ramsey_pulse
from src.qubit import TransmonQubit
from qutip import QobjEvo, basis, mesolve

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 6)})
print('Imports OK')

## 1. 实验设置

使用正交 Ramsey 序列（R_y(π/2) 和 R_{-y}(π/2)，τ=0），驱动频率固定在 sweet spot。改变 qubit flux bias 产生 detuning，通过测量 p_diff = (p_y - p_{-y})/2 和核函数 G_α = ∫k(t)dt 反推 Δω。

In [ ]:
f_at_0 = TransmonQubit(EC=0.2*2*np.pi, EJ=15*2*np.pi, T1=10000, T2=5000, n_levels=2, flux=0.0).frequency
omega_d = f_at_0
t_rabi = CONFIG.pulse.t_rabi.copy()
t_global = CONFIG.pulse.t_global.copy()

print(f'Drive frequency (sweet spot): {omega_d:.4f} rad·GHz')
print(f'Rabi pulse: {len(t_rabi)} pts, Global axis: {len(t_global)} pts')

In [ ]:
def run_measurement(flux_bias, order=3, n_amp_samples=7):
    """Run full Ramsey measurement at given flux bias.
    Returns: f_true, p_diff, G_coeffs, elapsed, t_kernel, kernels_diff
    """
    q_ref = TransmonQubit(EC=0.2*2*np.pi, EJ=15*2*np.pi, T1=10000, T2=5000, n_levels=2, flux=flux_bias)
    f_true = q_ref.frequency
    
    # Ramsey pulses
    ctrl_y = create_ramsey_pulse(t_rabi, tau=0.0, omega_d=omega_d, phase1=np.pi/2)
    ctrl_my = create_ramsey_pulse(t_rabi, tau=0.0, omega_d=omega_d, phase1=-np.pi/2)
    
    # Simulate p_diff
    q = TransmonQubit(EC=0.2*2*np.pi, EJ=15*2*np.pi, T1=10000, T2=5000, n_levels=2, flux=flux_bias)
    psi_e = basis(q.n_levels, 1)
    H_0 = QobjEvo(q.get_hamiltonian_rwa(omega_d))
    
    H_y = H_0 + QobjEvo(ctrl_y.hamiltonian_on(t_global), tlist=t_global, order=1)
    res_y = mesolve(H_y, q.state, t_global, [], e_ops=[psi_e*psi_e.dag()], options={'max_step': float(CONFIG.awg.dt)})
    p_y = float(res_y.expect[0][-1])
    
    H_my = H_0 + QobjEvo(ctrl_my.hamiltonian_on(t_global), tlist=t_global, order=1)
    res_my = mesolve(H_my, q.state, t_global, [], e_ops=[psi_e*psi_e.dag()], options={'max_step': float(CONFIG.awg.dt)})
    p_my = float(res_my.expect[0][-1])
    p_diff = (p_y - p_my) / 2.0
    
    # Kernel estimation
    qk = TransmonQubit(EC=0.2*2*np.pi, EJ=15*2*np.pi, T1=10000, T2=5000, n_levels=2, flux=flux_bias)
    e = KernelEstimator(mode='omega', method='exp', order=order, n_amp_samples=n_amp_samples, virtual_z_impl='math')
    t0 = time.time()
    r_y = e.estimate_full(ctrl_y, qk)
    r_my = e.estimate_full(ctrl_my, qk)
    elapsed = time.time() - t0
    
    k_diff = [(np.asarray(r_y.kernels[i]) - np.asarray(r_my.kernels[i])) / 2.0 for i in range(order)]
    G = [float(np.trapezoid(k, r_y.t_samples)) for k in k_diff]
    
    return f_true, p_diff, G, elapsed, r_y.t_samples, k_diff

print('measurement function ready')

## 2. 单点验证：不同 order 的 G 系数一致性

验证 order 1/2/3 给出的 G₁ = ∫k₁(t)dt 是否一致（应该一致——高阶拟合不应改变线性系数）。

In [ ]:
print('=== Single-point test at flux=0.03 ===')
for order, n_amp in [(1, 3), (2, 5), (3, 7)]:
    f_true, p_diff, G, elapsed, _, _ = run_measurement(0.03, order=order, n_amp_samples=n_amp)
    dw = p_diff / G[0] if abs(G[0]) > 1e-8 else 0.0
    f_meas = omega_d - dw
    err = abs(f_meas - f_true)
    detuning = abs(f_true - omega_d)
    print(f'Order {order}: G₁={G[0]:.4f}, error={err:.2e} GHz ({err/detuning*100:.1f}% of detuning), time={elapsed:.1f}s')
    for i in range(1, order):
        print(f'  |G{i+1}|/|G₁| = {abs(G[i])/max(abs(G[0]),1e-8):.4e}')

## 3. 核函数形状可视化

比较 order-3 下的差分核函数 k₁(t), k₂(t), k₃(t) 的形状。

In [ ]:
_, _, _, _, t_k, kerns = run_measurement(0.03, order=3, n_amp_samples=7)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles = ['k₁(t) — Linear response', 'k₂(t) — Quadratic (≈0, symmetry)', 'k₃(t) — Cubic response']
for i, ax in enumerate(axes):
    ax.plot(t_k, kerns[i], '.-', markersize=2, linewidth=0.8)
    ax.set_title(titles[i])
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel(f'k{i+1} (rad$^{{-{i+1}}}$)')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. 扫参：不同 flux 偏置下的频率估计误差

比较 order 1 和 order 3（含 Newton 修正）在大范围内的表现。

In [ ]:
flux_list = np.concatenate([
    np.linspace(0.005, 0.05, 10),
    np.linspace(0.06, 0.15, 6),
])
flux_list = np.unique(flux_list)

data = {'flux': [], 'detuning': [], 'p_diff': [],
        'err_o1': [], 'err_o3_newton': [],
        'G1': [], 'G2': [], 'G3': [], 'time': []}

for flux_bias in flux_list:
    f_true, p_diff, G, elapsed, _, _ = run_measurement(flux_bias, order=3, n_amp_samples=7)
    detuning = f_true - omega_d
    
    # Order-1: linear deconvolution
    dw1 = p_diff / G[0] if abs(G[0]) > 1e-8 else 0.0
    f1 = omega_d - dw1
    err1 = abs(f1 - f_true)
    
    # Order-3 Newton: solve p_diff = G₁·dw + (1/6)·G₃·dw³ 
    dw = dw1
    for _ in range(20):
        f_val = G[0]*dw + G[2]*dw**3/6.0 - p_diff
        f_prime = G[0] + G[2]*dw**2/2.0
        if abs(f_prime) < 1e-15: break
        dw_new = dw - f_val / f_prime
        if abs(dw_new - dw) < 1e-12: dw = dw_new; break
        dw = dw_new
    f3 = omega_d - dw
    err3 = abs(f3 - f_true)
    
    data['flux'].append(flux_bias)
    data['detuning'].append(detuning)
    data['p_diff'].append(p_diff)
    data['err_o1'].append(err1)
    data['err_o3_newton'].append(err3)
    data['G1'].append(G[0])
    data['G2'].append(G[1] if len(G) >= 2 else 0)
    data['G3'].append(G[2] if len(G) >= 3 else 0)
    data['time'].append(elapsed)

print(f'Scan complete: {len(flux_list)} points')

In [ ]:
flux_arr = np.array(data['flux'])
detuning_arr = np.array(data['detuning'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Frequency measurement error
ax = axes[0, 0]
ax.semilogy(abs(detuning_arr), data['err_o1'], 'o-', label='Order 1 (linear)', markersize=4)
ax.semilogy(abs(detuning_arr), data['err_o3_newton'], 's--', label='Order 3 (Newton cubic correction)', markersize=4)
ax.set_xlabel('|Detuning| (GHz)')
ax.set_ylabel('Frequency error (GHz)')
ax.set_title('Ramsey Frequency Measurement Error')
ax.legend()
ax.grid(True, alpha=0.3)

# Top-right: p_diff vs detuning
ax = axes[0, 1]
ax.plot(detuning_arr, data['p_diff'], 'o-', markersize=4)
G1_mean = np.mean(data['G1'])
ax.plot(detuning_arr, G1_mean * detuning_arr, '--', alpha=0.5, label=f'Linear (G₁={G1_mean:.2f})')
ax.set_xlabel('Detuning (GHz)')
ax.set_ylabel('p_diff')
ax.set_title('p_diff vs Detuning')
ax.legend()
ax.grid(True, alpha=0.3)

# Bottom-left: Higher-order kernel magnitudes
ax = axes[1, 0]
ax.semilogy(abs(detuning_arr), [abs(g)/max(abs(data['G1'][i]),1e-8) for i,g in enumerate(data['G2'])], 's-', label='|G₂|/|G₁|', markersize=4)
ax.semilogy(abs(detuning_arr), [abs(g)/max(abs(data['G1'][i]),1e-8) for i,g in enumerate(data['G3'])], '^-', label='|G₃|/|G₁|', markersize=4)
ax.set_xlabel('|Detuning| (GHz)')
ax.set_ylabel('Ratio')
ax.set_title('Higher-Order Kernel Relative Magnitude')
ax.legend()
ax.grid(True, alpha=0.3)

# Bottom-right: Computation time
ax = axes[1, 1]
ax.plot(abs(detuning_arr), data['time'], 'o-', markersize=4)
ax.set_xlabel('|Detuning| (GHz)')
ax.set_ylabel('Time (s)')
ax.set_title('Kernel Computation Time (order=3)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 线性 vs 三阶拟合：p_diff(Δω) 的非线性分析

将 p_diff 对 detuning 做多项式拟合，比较：
- **来自核函数**的 G₁, G₃（局部振幅扫描 → 对角 Volterra 核 → 积分）
- **来自 p_diff(Δω) 扫参**的有效系数（全局拟合）

这两者**不是同一个量**：核函数的 G₃ 测量的是在当前工作点的局部三阶响应，而 p_diff(Δω) 的非线性包含了 qubit 色散的非线性、脉冲动力学对 detuning 的非线性依赖等全局效应。

In [ ]:
from numpy.polynomial import polynomial as P

x = detuning_arr
y = np.array(data['p_diff'])

# Linear fit
c_lin = P.polyfit(x, y, deg=1)
y_lin = P.polyval(x, c_lin)

# Cubic fit (only odd terms due to symmetry)
c_cub = P.polyfit(x, y, deg=3)
y_cub = P.polyval(x, c_cub)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(x, y, 'o', markersize=4, label='Data')
ax1.plot(x, y_lin, '--', label=f'Linear: G₁(fit)={c_lin[1]:.4f}')
ax1.plot(x, y_cub, '-.', label=f'Cubic: G₁={c_cub[1]:.4f}, G₃/6={c_cub[3]:.4f}')
ax1.set_xlabel('Detuning (GHz)')
ax1.set_ylabel('p_diff')
ax1.set_title('p_diff vs Detuning: Linear vs Cubic Fit')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals
ax2.semilogy(abs(x), abs(y - y_lin), 'o-', markersize=4, label='Linear residual')
ax2.semilogy(abs(x), abs(y - y_cub), 's-', markersize=4, label='Cubic residual')
ax2.set_xlabel('|Detuning| (GHz)')
ax2.set_ylabel('Residual')
ax2.set_title('Fit Residuals')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Linear fit:  G₁ = {c_lin[1]:.4f}')
print(f'Cubic fit:   G₁ = {c_cub[1]:.4f},  G₃/6 = {c_cub[3]:.4f}  =>  G₃(eff) = {c_cub[3]*6:.4f}')
print(f'Kernel G₁:   {np.mean(data["G1"]):.4f}')
print(f'Kernel G₃:   {np.mean(data["G3"]):.2f}')
print()
print(f'G₁ match: kernel vs fit = {np.mean(data["G1"])/c_lin[1]:.4f}')
print(f'G₃ mismatch: kernel G₃={np.mean(data["G3"]):.1f} vs effective G₃={c_cub[3]*6:.1f}')

## 6. 结论与建议

| 发现 | 详情 |
|------|------|
| **G₁ 跨 order 一致** | order 1/2/3 给出的 G₁ 数值完全一致（差异 < 0.02%）——线性核函数在不同拟合阶数下稳定 |
| **k₂ ≈ 0（对称性）** | 正交 Ramsey 序列的对称性导致二阶响应抵消（\|G₂\|/\|G₁\| ~ 10⁻¹⁴），order 2 对测频无帮助 |
| **k₃ 显著但不等于有效三阶系数** | \|G₃\|/\|G₁\| ≈ 28，但是**局部**三阶响应。p_diff(Δω) 的全局三阶系数来自 qubit 色散非线性 + 脉冲动力学非线性，与核函数的局部 G₃ **不是同一个量** |
| **Newton 修正效果有限** | 在中 detuning 区（~0.12 GHz）有 ~7.6% 改善；小 detuning 无影响；大 detuning 反而可能恶化 |
| **测频误差的根本原因** | 不是核函数精度不足，而是 p_diff(Δω) 本身的非线性——大 detuning 下 p_diff 饱和甚至振荡，任何多项式修正都失效 |
| **线性安全区** | 当 \|Δω\| < ~0.05 GHz 时，线性模型误差 < 5% |
| **计算成本** | order 3 ≈ 4× order 1（3 s vs 0.8 s），但在安全区内精度无差异 |

### 核心洞察

**核函数的 Gₙ = ∫kₙ(t)dt 是"局部"量**——它测量的是在当前工作点附近，系统对 small perturbation 的 n 阶响应。

**p_diff(Δω) 的非线性是"全局"量**——当 detuning 从 0 扫到 -1.7 GHz 时，qubit 工作点大幅漂移，色散关系 ω(Φ) 的曲率、脉冲动力学对 detuning 的敏感度都在变化。

**两者不可混用**：用局部 G₃ 去修正全局 p_diff(Δω) 的立方项，类似于用一阶导数外推远处的函数值——在足够近时有效，远了必然发散。

### 建议

- **日常测频**：order=1 足够（快速、稳定），确保工作在 \|Δω\| < 0.05 GHz 的线性安全区内
- **大信号波形重建**：使用 Hammerstein-Volterra 反卷积（Phase 10.4），它用完整的 kₙ(t) 核函数族而非仅积分值
- **追求更高测频精度**：不应寄望于高阶核函数，而应使用闭环 Ramsey τ-sweep（`FrequencyMeasurement(method="ramsey")`），其精度不依赖线性近似